[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adan-rs/amd/blob/main/practicas/Practica_4b.ipynb)

# Práctica 4b: Estimación de la elasticidad ingreso del consumo

## Objetivo de aprendizaje
Aplicar una regresión lineal simple e interpretar correctamente
- La R cuadrada como medida de ajuste
- La significancia global del modelo
- La significancia individual del coeficiente
- La interpretación económica del coeficiente (elasticidad ingreso)

## Estructura del ejercicio
Esta práctica es una alternativa a la Práctica 4 (CAPM), orientada al análisis de mercados. Ambas cubren el mismo contenido estadístico, por lo que **debes elegir solo una de las dos**.

En la sección de "Contexto y preparación de datos" únicamente debes
- Sustituir el rubro de gasto por uno diferente al del ejemplo ('carnes')

En la sección de "Análisis e interpretación":
- Es donde debes escribir tu código adicional
- Es donde debes realizar la interpretación económica y estadística del modelo.

## Contexto y preparación de datos
Supongamos que trabajas en el área de inteligencia de mercados de una empresa de consumo y necesitas saber **qué tanto responde el consumo de una categoría de producto al ingreso del hogar**. La respuesta orienta decisiones concretas: si el consumo de la categoría apenas crece cuando el ingreso aumenta, el tamaño del mercado depende más del número de hogares que de su poder de compra, y la categoría es relativamente resistente a una recesión. Si el consumo crece más que proporcionalmente, el mercado se concentra en los hogares de mayor ingreso y queda expuesto al ciclo económico.

La medida estándar para responder esta pregunta es la *elasticidad ingreso de la demanda*: el cambio porcentual en el consumo ante un cambio de 1% en el ingreso.

$$
\eta = \frac{\%\Delta \text{consumo}}{\%\Delta \text{ingreso}}
$$

Para estimarla con una regresión, se suele suponer que la relación entre el gasto en la categoría y el ingreso del hogar tiene forma potencial:

$$
\text{gasto} = A \cdot \text{ingreso}^{\eta}
$$

Al aplicar logaritmos a ambos lados obtenemos una ecuación lineal que sí se puede estimar por mínimos cuadrados ordinarios:

$$
\ln(\text{gasto}) = \beta_0 + \eta \ln(\text{ingreso}) + \varepsilon
$$

La ventaja de esta transformación es que **el coeficiente de la variable independiente es directamente la elasticidad**: no requiere cálculos adicionales. Esta relación entre gasto e ingreso se conoce como *curva de Engel*.

Como referencia para la interpretación:

| Valor de $\eta$ | Tipo de bien | Lectura |
|---|---|---|
| $\eta < 0$ | Bien inferior | El consumo disminuye cuando el ingreso aumenta |
| $0 < \eta < 1$ | Bien normal, necesidad | El consumo crece, pero menos que proporcionalmente (inelástico) |
| $\eta > 1$ | Bien normal, superior o de lujo | El consumo crece más que proporcionalmente (elástico) |

> **Nota: alcance didáctico del ejercicio.** El modelo que estimarás aquí es una versión simplificada con fines de aprendizaje. Antes de interpretar los resultados, ten presente que:
> - Se trata de una elasticidad **ingreso**, no de una elasticidad **precio**. Con los datos de una sola encuesta no existe variación de precios que permita identificar la respuesta del consumo al precio.
> - El archivo es una submuestra preparada para el curso y **no incluye el factor de expansión** de la ENIGH, por lo que los resultados no son estimaciones representativas de la población. El propósito es practicar la estimación e interpretación, no producir cifras oficiales.
> - La unidad de observación es el **hogar**, no el individuo, y el gasto corresponde al periodo de referencia de la encuesta.
> - El modelo es descriptivo: mide asociación, no efecto causal. El ingreso no fue asignado aleatoriamente entre los hogares.
>
> Estas limitaciones no invalidan el ejercicio, pero sí acotan las conclusiones que puedes defender.

In [ ]:
# Importar bibliotecas
import pandas as pd
import numpy as np
import statsmodels.api as sm

Utilizaremos el archivo `enigh2020.xlsx`, que contiene información de hogares del área metropolitana de Monterrey a partir de la Encuesta Nacional de Ingresos y Gastos de los Hogares (ENIGH) 2020 del INEGI. Las dos variables centrales son:
- `ing_cor`: ingreso corriente total del hogar
- `gasto_mon`: gasto monetario total del hogar

Y los rubros de gasto disponibles son: `alimentos`, `cereales`, `carnes`, `leche`, `huevo`, `bebidas`, `agua`, `energia` y `cuida_pers`.

**Importante: utiliza un rubro de gasto diferente al del ejemplo ('carnes')**

In [ ]:
# Parámetros
rubro = 'carnes'    # Cambiar por otro rubro de gasto

# Importar el archivo "data/enigh2020.xlsx"
df = pd.read_excel('https://github.com/adan-rs/amd/raw/main/data/enigh2020.xlsx')

In [ ]:
# Revisar las variables y el número de observaciones
df.info()

In [ ]:
# Estadística descriptiva de las variables de interés
df[['ing_cor', 'gasto_mon', rubro]].describe().T

El modelo se estima con logaritmos, y el logaritmo de cero no está definido. Por ello es necesario conservar únicamente los hogares con ingreso positivo y con gasto positivo en el rubro elegido. Este filtro no es un detalle técnico menor: **cambia la población sobre la que se estima la elasticidad**, y en la interpretación deberás decir cuántos hogares quedaron fuera y qué implica.

Los pasos son:
1. Filtrar los hogares con `ing_cor > 0` y con gasto en el rubro mayor a cero
2. Crear las variables en logaritmo natural
3. Revisar cuántas observaciones quedaron

In [ ]:
# Filtrar hogares con ingreso y gasto positivos
datos = df[(df['ing_cor'] > 0) & (df[rubro] > 0)].copy()

# Crear variables en logaritmo natural
datos['log_ingreso'] = np.log(datos['ing_cor'])
datos['log_gasto'] = np.log(datos[rubro])

# Revisar el efecto del filtro
print(f'Observaciones originales: {len(df)}')
print(f'Observaciones utilizadas: {len(datos)}')
print(f'Observaciones excluidas:  {len(df) - len(datos)}')

datos[['ing_cor', rubro, 'log_ingreso', 'log_gasto']].head()

## Análisis e interpretación
Estima un modelo de regresión con `log_gasto` como variable dependiente y `log_ingreso` como variable independiente (no olvides incluir el intercepto). Se sugiere acompañar la estimación con un diagrama de dispersión de ambas variables en logaritmos.

Una vez estimada la regresión, responde con explicación en lenguaje claro (no solo copiar valores):

- *El ajuste del modelo (R cuadrada).* ¿Qué porcentaje de la variación del gasto en la categoría es explicado por el modelo? ¿Consideras que el ajuste es alto, moderado o bajo? Toma en cuenta que se trata de microdatos de hogares en corte transversal: ¿qué otros factores además del ingreso podrían explicar cuánto gasta un hogar en esta categoría? ¿Un ajuste bajo invalida el coeficiente estimado, o son dos cosas distintas?
- *El p-valor del estadístico F.* ¿El modelo en conjunto es estadísticamente significativo? ¿Qué hipótesis está evaluando el estadístico F? Interpreta el p-valor en términos prácticos considerando un nivel de significancia del 5%.
- *El p-valor de la variable independiente.* ¿La elasticidad es estadísticamente diferente de cero? ¿Qué implicaría en términos de mercado que no fuera significativa?
- *El coeficiente de regresión (elasticidad ingreso).* Interpreta el valor estimado en términos porcentuales: si el ingreso de un hogar aumenta 10%, ¿cuánto aumenta su gasto en esta categoría? ¿La categoría es inelástica o elástica al ingreso? ¿Se comporta como una necesidad o como un bien superior? Con base en ello, ¿en qué segmento de ingreso ubicarías la oportunidad comercial y qué tan expuesta está la categoría a una caída generalizada del ingreso?
- *Las decisiones de preparación de datos.* ¿Cuántos hogares quedaron fuera al excluir los que reportan gasto cero? ¿Qué puede significar que un hogar reporte cero en esta categoría? ¿La elasticidad que estimaste aplica a todos los hogares o solo a quienes ya consumen la categoría?

## Uso de IA generativa (para extender, no para resolver)

La IA se usa para **ampliar** la práctica partiendo de lo que ya construiste, no para hacerla. Declara la herramienta y la versión utilizada (por ejemplo, ChatGPT 5, Claude Opus 4.5, Gemini 3 Pro, Copilot).

**Qué debes entregar** (cuatro bloques, en celdas de texto dentro del notebook):

1. **Tu pregunta de negocio.** Una pregunta propia, pertinente y que la práctica **no** responda. Formúlala en primera persona: "Quiero saber si...", "Me preocupa que...". No se acepta reproducir la pregunta del ejercicio ni preguntas genéricas. Del tipo esperado (no para copiar): *"Quiero saber si la elasticidad que estimé es distinta entre los hogares con jefatura femenina y masculina, porque eso cambiaría a quién dirigir la categoría"*.
2. **El prompt completo, transcrito en celda de texto** (no de código). Debe incluir: rol, objetivo, tu código, **los resultados reales de tu regresión pegados** (la salida de `summary()`) y una restricción explícita de lo obvio (por ejemplo: "no me expliques qué es una elasticidad ni qué es un p-valor"). Un prompt de una línea, sin contexto ni resultados, no cuenta.
3. **Qué adopté y qué descarté.** De la respuesta recibida, indica qué implementaste y qué dejaste fuera, con la razón. La implementación usa **máximo 50 líneas de código**.
4. **Verificación.** Un cálculo, contraejemplo o comprobación que valide o refute algo que la IA afirmó. Por ejemplo: si la IA afirma que tu categoría se comporta como un bien de lujo, contrástalo con tu coeficiente y su intervalo de confianza; si sugiere que unos pocos hogares de ingreso muy alto sostienen el resultado, reestima sin ellos y compara.

**Evidencia auditable**: pega el resultado completo de la extensión (texto, tablas o código), no un enlace a la conversación. Revisa que el documento o el código no quede cortado.

**No se acepta**:
- Transferir las instrucciones de la práctica a la IA, ya sea copiándolas o parafraseándolas, para que ella la resuelva.
- Usar la IA como enciclopedia: respuestas generales sin tus datos ni tu código. Está bien usarla para comprender, pero debe haber contenido propio nuevo.
- Transcribir sugerencias sin implementarlas, o dar por cierto lo que la IA afirma sin comprobarlo.
- Delegar la interpretación: las conclusiones y su redacción son tuyas.

**Buenas prácticas sugeridas**: repreguntar a la IA sobre su propia respuesta; pedirle explícitamente las limitaciones de lo que propone; traer un concepto externo al curso y aplicarlo a tus datos.

**Defensa oral**: cualquier práctica puede ser seleccionada al azar para una defensa oral breve (3 a 5 minutos), en la que deberás explicar tus decisiones, tu código y tus conclusiones. Un trabajo que no pueda ser explicado por su autor se considerará evidencia de trabajo no auténtico y podrá ser penalizado.

## Entregable
Notebook en Jupyter exportado a pdf o html, con el código, análisis e interpretación.

## Rúbrica de evaluación
1. Implementación técnica (20%): importación correcta de datos, filtrado y transformación logarítmica adecuados, modelo estimado correctamente, y justificación explícita de las decisiones de preparación de datos.
2. Interpretación de R cuadrada (10%): interpreta correctamente y discute la implicación de trabajar con microdatos de hogares. Se penalizará que sólo reporte el número sin interpretación real.
3. Significancia del modelo (10%): Interpreta p-valor y concluye correctamente. Se penalizará que sólo indique si es significativo o no.
4. Significancia del coeficiente (10%): Interpreta p-valor y conecta con el significado de mercado. Se penalizará que sólo indique si es significativo o no.
5. Interpretación económica de la elasticidad (15%): Interpreta el coeficiente en términos porcentuales, clasifica correctamente el tipo de bien y deriva una implicación comercial. Se penalizará que sólo describa el número.
6. Claridad y redacción (15%): Explicación clara, lenguaje técnico correcto, coherencia. Se penalizará respuestas vagas o confusas.
7. **Extensión del análisis con IA generativa (20%)**, evaluada en cuatro partes iguales (5% cada una):
   - *Pregunta propia (5%)*: pregunta de negocio formulada por el alumno, pertinente y no respondida por la práctica. Se anula si reproduce la pregunta del ejercicio o si es genérica.
   - *Calidad del prompt (5%)*: transcrito completo en celda de texto, con rol, objetivo, código y los resultados reales de la regresión, más una restricción explícita de lo obvio. Se anula si se transfieren las instrucciones de la práctica (copiadas o parafraseadas).
   - *Ejecución: qué adopté y qué descarté (5%)*: lo propuesto se implementa (máximo 50 líneas de código) y se explica qué quedó fuera y por qué. No basta transcribir sugerencias.
   - *Verificación e interpretación propia (5%)*: un cálculo, contraejemplo o comprobación que valide o refute una afirmación de la IA, y conclusiones redactadas por el alumno.

   *Requisito de forma*: la evidencia debe ser auditable, esto es, resultado completo pegado en el notebook, sin cortes y sin enlaces a la conversación como único respaldo.